# Explore alignment between GCM and CPM

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
cpm_dataset="demo-ccpm_pr"
gcm_dataset="demo-gcm_pr"
split="test"
ensemble_members = ensemble_members = [
    "01",
    "04",
    "05",
    "06",
    "07",
    "08",
    "09",
    "10",
    "11",
    "12",
    "13",
    "15",
]
var = "target_pr"

In [ ]:
import functools
import math
import string

import cftime
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_utils import cp_model_rotated_pole, DatasetMetadata
from mlde_analysis import plot_map, VAR_LABELS
from mlde_analysis.data import open_dataset_split

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
cpm_da = open_dataset_split(cpm_dataset, split, ensemble_members)[var].expand_dims(simulation=["CPM"])
gcm_da = open_dataset_split(gcm_dataset, split, ensemble_members)[var].expand_dims(simulation=["GCM"])
DA = xr.concat([cpm_da, gcm_da], dim="simulation")
DA

In [ ]:
example_queries = [
    dict(ensemble_member="04", time=cftime._cftime.Datetime360Day.strptime("2035-12-12 12:00:00", "%Y-%m-%d %H:%M:%S", calendar="360_day")),
    dict(ensemble_member="09", time=cftime._cftime.Datetime360Day.strptime("2075-01-16 12:00:00", "%Y-%m-%d %H:%M:%S", calendar="360_day")),
    dict(ensemble_member="12", time=cftime._cftime.Datetime360Day.strptime("2035-08-25 12:00:00", "%Y-%m-%d %H:%M:%S", calendar="360_day")),
    dict(ensemble_member="12", time=cftime._cftime.Datetime360Day.strptime("2035-08-19 12:00:00", "%Y-%m-%d %H:%M:%S", calendar="360_day")),
]

fig = plt.figure(layout="constrained", figsize=(3, 6))
    
axes = fig.subplots(len(example_queries), 2, subplot_kw={"projection": cp_model_rotated_pole})

for iq, q in enumerate(example_queries):    
    for isim in range(2):
        example_da = DA.sel(ensemble_member=q["ensemble_member"]).sel(time=q["time"], method="nearest").isel(simulation=isim)
        ax = axes[iq][isim]
        pcm = plot_map(example_da, ax=ax, style="pr")
        if iq == 0:
            ax.set_title(example_da["simulation"].item())

cb = fig.colorbar(
    pcm,
    ax=axes,
    location="bottom",
    orientation="horizontal",
    shrink=0.8,
    extend="both",
)
cb.ax.tick_params(axis="both", which="major", labelsize="small")
cb.set_label(VAR_LABELS["pr"], fontsize="small")

plt.show()